In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_dominance(matrix):
    """Checks for strictly dominant strategies in a 2x2 matrix."""
    # Row Player (Player 1)
    p1_dominant = None
    if matrix[0,0][0] > matrix[1,0][0] and matrix[0,1][0] > matrix[1,1][0]:
        p1_dominant = "Strategy 0 (Top)"
    elif matrix[1,0][0] > matrix[0,0][0] and matrix[1,1][0] > matrix[0,1][0]:
        p1_dominant = "Strategy 1 (Bottom)"
        
    # Column Player (Player 2)
    p2_dominant = None
    if matrix[0,0][1] > matrix[0,1][1] and matrix[1,0][1] > matrix[1,1][1]:
        p2_dominant = "Strategy 0 (Left)"
    elif matrix[0,1][1] > matrix[0,0][1] and matrix[1,1][1] > matrix[1,0][1]:
        p2_dominant = "Strategy 1 (Right)"
        
    return p1_dominant, p2_dominant

def find_nash(matrix):
    """Finds pure strategy Nash Equilibria."""
    equilibria = []
    for r in range(2):
        for c in range(2):
            # Check if Player 1 is playing best response to Player 2
            p1_best = matrix[r,c][0] >= max(matrix[:, c, 0])
            # Check if Player 2 is playing best response to Player 1
            p2_best = matrix[r,c][1] >= max(matrix[r, :, 1])
            
            if p1_best and p2_best:
                equilibria.append((r, c))
    return equilibria

In [2]:
# Payoff Matrix: (USA, USSR)
# 0: Disarm, 1: Arm
# Payoffs: 3 (Great), 2 (Good), 1 (Bad), 0 (Catastrophic)
cold_war_matrix = np.array([
    [(2, 2), (0, 3)],  # Disarm, Disarm | Disarm, Arm
    [(3, 0), (1, 1)]   # Arm, Disarm    | Arm, Arm
], dtype=object)

p1_dom, p2_dom = find_dominance(cold_war_matrix)
nash_eq = find_nash(cold_war_matrix)

print("--- COLD WAR ANALYSIS ---")
print(f"USA Dominant Strategy: {p1_dom}")
print(f"USSR Dominant Strategy: {p2_dom}")
print(f"Nash Equilibrium (Indices): {nash_eq} (Both Arm)")

--- COLD WAR ANALYSIS ---
USA Dominant Strategy: Strategy 1 (Bottom)
USSR Dominant Strategy: Strategy 1 (Right)
Nash Equilibrium (Indices): [(1, 1)] (Both Arm)


In [3]:
def bayesian_expected_utility(action, prob_rational, matrix_rational, matrix_aggressive):
    """Calculates expected utility for USA based on the perceived type of USSR."""
    # Action 0: USA Disarms, Action 1: USA Arms
    # Assume USSR will play their dominant strategy for their type
    # Rational USSR (Type A) plays 'Arm' (Strategy 1)
    # Aggressive USSR (Type B) plays 'Arm' (Strategy 1)
    
    # In this specific Dilemma, both types arm, so the result is the same.
    # But AToM-Net uses this to detect if the 'Aggressive' probability is spiking.
    eu = (prob_rational * matrix_rational[action, 1][0]) + \
         ((1 - prob_rational) * matrix_aggressive[action, 1][0])
    return eu

# Let's see how USA's thinking changes if they think USSR is 90% likely to be Aggressive
prob_rational = 0.1 
print(f"USA Expected Utility if they Arm: {bayesian_expected_utility(1, prob_rational, cold_war_matrix, cold_war_matrix)}")

USA Expected Utility if they Arm: 1.0


In [4]:
class AtomNetValidator:
    def __init__(self):
        self.deception_risk = 0.0
        self.belief_honesty = 0.9

    def validate(self, stated_intent, actual_action):
        """
        Stage 2: Economic Consistency Check
        If intent is 'Peaceful' but action is 'Arming', contradiction = 1.
        """
        contradiction = 1 if (stated_intent == "Peaceful" and actual_action == "Arm") else 0
        
        # Bayesian update (Simplified)
        if contradiction:
            self.belief_honesty *= 0.2 # Alpha penalty
            self.deception_risk = (0.7 * self.deception_risk) + 0.3 # Beta EMA update
            
        return self.belief_honesty, self.deception_risk

# Scenario: USSR says "We want peace" then builds a missile silo.
validator = AtomNetValidator()
h, r = validator.validate("Peaceful", "Arm")

print("--- ATOM-NET COGNITIVE MONITOR ---")
print(f"Inferred Honesty: {h:.2f}")
print(f"Deception Risk: {r:.2f}")

--- ATOM-NET COGNITIVE MONITOR ---
Inferred Honesty: 0.18
Deception Risk: 0.30


In [5]:
import numpy as np

class TragedyOfCommons_BeliefEngine:
    def __init__(self):
        # Hidden state: Is the opponent a "Cooperator" or an "Exploiter"?
        self.prob_exploiter = 0.5 
        
    def update_belief(self, opponent_action):
        """Bayesian update based on observed resource usage."""
        # Likelihoods: P(Overuse | Exploiter) = 0.9, P(Overuse | Cooperator) = 0.2
        if opponent_action == "Overuse":
            likelihood_exp = 0.9
            likelihood_coop = 0.2
        else:
            likelihood_exp = 0.1
            likelihood_coop = 0.8
            
        numerator = likelihood_exp * self.prob_exploiter
        denominator = numerator + (likelihood_coop * (1 - self.prob_exploiter))
        self.prob_exploiter = numerator / denominator
        return self.prob_exploiter

# Simulation
atom_brain = TragedyOfCommons_BeliefEngine()
print("--- TRAGEDY OF THE COMMONS: PMD RESOURCE TRACKING ---")
history = ["Share", "Overuse", "Overuse"]

for turn, action in enumerate(history):
    p_exploiter = atom_brain.update_belief(action)
    print(f"Turn {turn+1}: Opponent plays '{action}'. AToM-Net belief they are an Exploiter: {p_exploiter*100:.1f}%")

--- TRAGEDY OF THE COMMONS: PMD RESOURCE TRACKING ---
Turn 1: Opponent plays 'Share'. AToM-Net belief they are an Exploiter: 11.1%
Turn 2: Opponent plays 'Overuse'. AToM-Net belief they are an Exploiter: 36.0%
Turn 3: Opponent plays 'Overuse'. AToM-Net belief they are an Exploiter: 71.7%


In [6]:
class Cournot_EconomicValidator:
    def __init__(self):
        self.deception_risk = 0.0
        
    def validate_market_bluff(self, stated_capacity, raw_materials_purchased):
        """
        Stage 2: C(action, belief)
        Checks if the competitor's PR matches their actual supply chain.
        """
        # If they claim High capacity but buy Low materials -> Contradiction!
        contradiction = 1 if (stated_capacity == "High" and raw_materials_purchased == "Low") else 0
        
        if contradiction == 1:
            # Beta EMA update for deception
            beta = 0.7
            self.deception_risk = (beta * self.deception_risk) + ((1 - beta) * contradiction)
            
        return self.deception_risk

# Simulation
validator = Cournot_EconomicValidator()
print("\n--- COURNOT DUOPOLY: COMPETITOR BLUFF DETECTION ---")

# Turn 1: Competitor claims High capacity, buys High materials (Consistent)
risk = validator.validate_market_bluff("High", "High")
print(f"Turn 1 (Consistent): Deception Risk is {risk:.2f}")

# Turn 2: Competitor claims High capacity (to scare us), but buys Low materials (Bluff!)
risk = validator.validate_market_bluff("High", "Low")
print(f"Turn 2 (Caught Bluff!): Deception Risk spikes to {risk:.2f}")

# AToM-Net's Decision Engine now knows to ignore their "Cheap Talk" and maintain high production!
if risk > 0.25:
    print("AToM-Net Action: Ignore competitor threats. Maintain maximum production output.")


--- COURNOT DUOPOLY: COMPETITOR BLUFF DETECTION ---
Turn 1 (Consistent): Deception Risk is 0.00
Turn 2 (Caught Bluff!): Deception Risk spikes to 0.30
AToM-Net Action: Ignore competitor threats. Maintain maximum production output.


In [7]:
class FOA_DecisionEngine:
    def __init__(self):
        # The arbitrator's estimated "fair value" is around $100k
        self.arbitrator_estimate = 100 
        
    def calculate_optimal_bid(self, opponent_inferred_bid_range):
        """
        Stage 3: Expected Utility Maximization over continuous space.
        Finds the highest possible bid that still beats the opponent's expected bid.
        """
        best_eu = 0
        best_bid = 0
        
        # We test bids from $90k to $130k
        for my_bid in range(90, 131):
            # Probability we win is based on being closer to the arbitrator's estimate
            # than the opponent. We simulate against all likely opponent bids.
            win_prob = 0
            for opp_bid in opponent_inferred_bid_range:
                my_distance = abs(my_bid - self.arbitrator_estimate)
                opp_distance = abs(opp_bid - self.arbitrator_estimate)
                
                if my_distance < opp_distance:
                    win_prob += (1.0 / len(opponent_inferred_bid_range))
            
            # EU = Probability of Winning * The Payoff (Our Bid)
            eu = win_prob * my_bid
            
            if eu > best_eu:
                best_eu = eu
                best_bid = my_bid
                
        return best_bid, best_eu

# Simulation
foa_engine = FOA_DecisionEngine()
print("\n--- FINAL OFFER ARBITRATION: EU MAXIMIZATION ---")
# AToM-Net infers the opponent is greedy and will ask for $110k to $120k
inferred_opp_bids = list(range(110, 121)) 

optimal_bid, expected_value = foa_engine.calculate_optimal_bid(inferred_opp_bids)
print(f"Opponent is predicted to bid between ${inferred_opp_bids[0]}k and ${inferred_opp_bids[-1]}k.")
print(f"AToM-Net Optimal Bid: ${optimal_bid}k (Expected Utility: ${expected_value:.1f}k)")


--- FINAL OFFER ARBITRATION: EU MAXIMIZATION ---
Opponent is predicted to bid between $110k and $120k.
AToM-Net Optimal Bid: $109k (Expected Utility: $109.0k)
